# Análisis Geoespacial IPM - Zona de Estudio: Roosevelt

Este notebook prioriza la visualización cartográfica de las 5 variables más relevantes del Índice de Pobreza Multidimensional (IPM) en el corredor Roosevelt (Buffer 100m).

**Características:**
- Mapa base con delimitación de manzanas.
- Etiquetas de valores dentro de cada manzana.
- Paleta de colores accesible para personas daltónicas (Viridis).
- Leyenda de categorías basada en la incidencia.

In [ ]:
# 1. Instalación de dependencias (solo en Colab)
import sys
if 'google.colab' in sys.modules:
    !pip install geopandas matplotlib seaborn -q

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style('white')
print('Librerías listas')

In [ ]:
# 2. Carga de datos
# Ruta relativa al repositorio
PATH_GEOJSON = '../data/Geojson_Roosevelt/geojson_filtrado_tramos_Roosevelt_Buffer_100/Mzn_ipm_variables_filtrado_tramos_Roosevelt_Buffer_100.geojson'

if not os.path.exists(PATH_GEOJSON):
    print("Archivo no encontrado. Ajustando ruta para Colab...")
    PATH_GEOJSON = 'Pobreza_multidimensional_y_condicion_social/indice_Pobreza/data/Geojson_Roosevelt/geojson_filtrado_tramos_Roosevelt_Buffer_100/Mzn_ipm_variables_filtrado_tramos_Roosevelt_Buffer_100.geojson'

gdf = gpd.read_file(PATH_GEOJSON)
print(f'Datos cargados: {len(gdf)} manzanas')

In [ ]:
# 3. Identificación de las 15 variables y selección de las Top 5
variables_ipm = [
    'ANALF_', 'BAJO_', 'INFANCIA_', 'INASIS_', 'REZAGO_',
    'TRAB_INFAN', 'DEPEN_', 'INFOR_', 'SALUD_', 'ASEGU_',
    'HACI_', 'PARED_', 'EXCRE_', 'PISOS_', 'AGUA_'
]

# Diccionario para nombres legibles
diccionario_nombres = {
    'ANALF_': 'Analfabetismo', 'BAJO_': 'Bajo Logro Educativo', 'INFANCIA_': 'Barreras Infancia',
    'INASIS_': 'Inasistencia Escolar', 'REZAGO_': 'Rezago Escolar', 'TRAB_INFAN': 'Trabajo Infantil',
    'DEPEN_': 'Dependencia Económica', 'INFOR_': 'Informalidad', 'SALUD_': 'Sin Aseguramiento Salud',
    'ASEGU_': 'Barreras Salud', 'HACI_': 'Hacinamiento Crítico', 'PARED_': 'Paredes Inadecuadas',
    'EXCRE_': 'Excretas Inadecuadas', 'PISOS_': 'Pisos Inadecuados', 'AGUA_': 'Sin Agua Mejorada'
}

# Calcular promedio local para elegir las 5 más relevantes (mayor incidencia)
top_5 = gdf[variables_ipm].mean().sort_values(ascending=False).head(5).index.tolist()
print('Las 5 variables con mayor incidencia en Roosevelt son:')
for v in top_5:
    print(f'- {diccionario_nombres[v]} ({v})')

In [ ]:
# 4. Función de mapeo optimizada para accesibilidad
def plot_ipm_variable(gdf, column, title):
    fig, ax = plt.subplots(1, 1, figsize=(15, 12))
    
    # Dibujar manzanas
    gdf.plot(column=column, 
             cmap='viridis', 
             legend=True, 
             scheme='NaturalBreaks',
             k=5,
             edgecolor='black', 
             linewidth=0.5,
             alpha=0.8,
             legend_kwds={'title': f'% Hogares con {title}', 'loc': 'lower right'},
             ax=ax)
    
    # Agregar etiquetas de valor dentro de la manzana
    for x, y, label in zip(gdf.geometry.centroid.x, gdf.geometry.centroid.y, gdf[column]):
        if label > 0:
            ax.annotate(f'{label:.1f}', 
                        xy=(x, y), 
                        xytext=(0, 0), 
                        textcoords='offset points', 
                        ha='center', 
                        fontsize=8, 
                        fontweight='bold', 
                        color='white', 
                        path_effects=[plt.matplotlib.patheffects.withStroke(linewidth=2, foreground='black')])
    
    ax.set_title(f'Incidencia de {title} - Roosevelt', fontsize=16, fontweight='bold')
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()

In [ ]:
# 5. Generar los 5 mapas prioritarios
for var in top_5:
    plot_ipm_variable(gdf, var, diccionario_nombres[var])